# Parallelism in multislice calculations #

This library uses JAX for multislice calculations. These calculations can be performed on CPUs, GPUs, and TPUs.
They leverage both device-level parallelism and thread-level parallelism within the processors, enabling high-speed execution of multislice computations.
However, the implementation of device-level parallelism differs among CPUs, GPUs, and TPUs.

|Processor|Number of devices (typical)|
|:-:|:-:|
|CPU|1|
|GPU|Number of GPU chips|
|TPU|Number of TPU cores|

Device-level parallelism is determined by this number.
When using a CPU, the number of devices is typically recognized as 1, so device-level parallelism is not applied; only thread-level parallelism is used.

The following codes are intended to run on a CPU.

In [1]:
import jax
print("Devices:", jax.devices())

Devices: [CpuDevice(id=0)]


In a CPU environment, JAX typically recognizes only a single device.

In [2]:
import numpy as np
from lys_mat import CrystalStructure
from lys_em import FunctionSpace, CrystalPotential, TEM, TEMParameter, multislice, diffraction
import time

crys = CrystalStructure.loadFrom("data/Au.cif")
sp = FunctionSpace.fromCrystal(crys, 128, 128, 50)
pot = CrystalPotential(sp, crys)
tem = TEM(60e3, params=[TEMParameter(tilt=[2, phi]) for phi in np.arange(0, 360, 360 / 900)])

start = time.time()
data = diffraction(multislice(pot, tem)).sum(axis=0).block_until_ready()
print("1st calculation time:", f"{time.time()-start:.2f}", "s")

start = time.time()
data = diffraction(multislice(pot, tem)).sum(axis=0).block_until_ready()
print("2nd calculation time:", f"{time.time()-start:.2f}", "s")

1st calculation time: 14.93 s
2nd calculation time: 17.74 s


When performing calculations on a CPU, only thread-level parallelism is typically used, so the computation speed is relatively low.

<span style="color:red;font-size:130%;">**Now, restart the kernel (<img src="data/restart.png" width="25">) and execute the following code:**</span>

By using *jax.config.update("jax_num_cpu_devices", number)*, you can change the number of devices recognized by JAX, even when running on a CPU. This command must be executed immediately after importing JAX. Doing so enables device-level parallelism, allowing high-speed calculations even on a CPU.

In [1]:
import jax
jax.config.update("jax_num_cpu_devices", 8)   # run immediately after importing JAX 
print("Devices:", jax.devices())

Devices: [CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]


In [2]:
import numpy as np
from lys_mat import CrystalStructure
from lys_em import FunctionSpace, CrystalPotential, TEM, TEMParameter, multislice, diffraction
import time

crys = CrystalStructure.loadFrom("data/Au.cif")
sp = FunctionSpace.fromCrystal(crys, 128, 128, 50)
pot = CrystalPotential(sp, crys)
tem = TEM(60e3, params=[TEMParameter(tilt=[2, phi]) for phi in np.arange(0, 360, 360 / 900)])

start = time.time()
data = diffraction(multislice(pot, tem)).sum(axis=0).block_until_ready()
print("1st calculation time:", f"{time.time()-start:.2f}", "s")

start = time.time()
data = diffraction(multislice(pot, tem)).sum(axis=0).block_until_ready()
print("2nd calculation time:", f"{time.time()-start:.2f}", "s")

1st calculation time: 4.63 s
2nd calculation time: 3.50 s


Calculations are executed quickly using device-level parallelism. Additionally, JAX compiles the first computation, so subsequent calculations run even faster.